---
title: "Part 7: Data Storytelling and House Style"
---

**DS-MLOps Python Foundations**

**Python 3.12+ | Author: Anthony Faustine**

## Before you begin

This notebook assumes you have completed Part 5 (`05-matplotlib.ipynb`) and Part 6 (`06-lets-plot.ipynb`). Parts 5 and 6 taught you how to make a chart at all. This part is about taste: which chart to make, what to leave out of it, and how to brand it once it is ready to show someone other than yourself, using this project's own `ark.plot` module.

| Topic | Why it matters |
|---|---|
| **Chart selection** | The question decides the chart, not the data type |
| **Data-ink ratio** | Every pixel that is not data is a tax on the reader's attention |
| **Common chart lies** | The same data, drawn to mislead instead of inform |
| **Tables vs. charts** | Exact lookup wants a table; pattern recognition wants a chart |
| **House style** | Replacing per-chart styling with one reusable theme |

## Callout guide

<table style='border-collapse:collapse;width:60%'>
<tr><td style='padding:6px 12px;background:#EAF3FA;border-left:4px solid #0369A1'><i class="bi bi-info-circle-fill" style="color:#0369A1"></i> <b>Key Concept</b></td><td style='padding:6px 12px'>Core idea with precise definition</td></tr>
<tr><td style='padding:6px 12px;background:#EAF7F0;border-left:4px solid #059669'><i class="bi bi-journal-code" style="color:#059669"></i> <b>Example</b></td><td style='padding:6px 12px'>Worked illustration</td></tr>
<tr><td style='padding:6px 12px;background:#FFF1E6;border-left:4px solid #EA580C'><i class="bi bi-puzzle-fill" style="color:#EA580C"></i> <b>Activity</b></td><td style='padding:6px 12px'>Hands-on challenge to try yourself</td></tr>
<tr><td style='padding:6px 12px;background:#FEF2F2;border-left:4px solid #DC2626'><i class="bi bi-bug-fill" style="color:#DC2626"></i> <b>Common Mistake</b></td><td style='padding:6px 12px'>Bug pattern to watch for</td></tr>
<tr><td style='padding:6px 12px;background:#F4F5F6;border-left:4px solid #6B7280'><i class="bi bi-lightbulb-fill" style="color:#6B7280"></i> <b>Pro Tip</b></td><td style='padding:6px 12px'>Production-grade habit</td></tr>
</table>

## Learning Objectives

By the end of Part 7 you will be able to:

| # | Skill | Covered in |
|---|---|---|
| 1 | Choose a chart type based on the question, not habit | Sec. 1 |
| 2 | Recognise and avoid the most common ways charts mislead | Sec. 2 |
| 3 | Decide when a table communicates better than a chart | Sec. 3 |
| 4 | Apply `ark.plot`'s house theme to matplotlib and lets-plot charts | Sec. 4 |
| 5 | Produce one polished, captioned chart from a messy dataset | Capstone |

## 1. Picking the Right Chart, and Cutting What Does Not Help

The most common visualisation mistake is not a styling problem, it is a **selection** problem: building a chart type out of habit instead of asking what question it needs to answer. A pie chart cannot answer "which course improved the most," because comparing angles is something people are measurably worse at than comparing positions along a shared axis. That is not opinion, it is the finding of a well-known study by Cleveland and McGill (1984) ranking how accurately people read different visual encodings: position along a common scale first, then length, then angle, then area, with colour saturation last.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

courses = np.array(["Machine Learning", "Data Structures", "Statistics"])
semesters = np.array(["Fall 2024", "Spring 2025"])

n_per_group = 60
course_col = np.repeat(courses, n_per_group * len(semesters))
semester_col = np.tile(np.repeat(semesters, n_per_group), len(courses))

course_base = {"Machine Learning": 68, "Data Structures": 74, "Statistics": 71}
semester_bump = {"Fall 2024": 0, "Spring 2025": 4}

exam_score = np.array(
    [rng.normal(course_base[c] + semester_bump[s], 10) for c, s in zip(course_col, semester_col, strict=True)]
).clip(0, 100)

results = pd.DataFrame({"course": course_col, "semester": semester_col, "exam_score": exam_score})
results.head()

**Data-ink ratio**, a term from Edward Tufte's *The Visual Display of Quantitative Information*, is the proportion of a chart's ink that represents actual data, as opposed to gridlines, borders, redundant legends, and other decoration. Maximising it does not mean a bare chart, it means every mark earns its place. Compare matplotlib's defaults to a version with the same data and nothing extra:

In [ ]:
import matplotlib.pyplot as plt

course_means = results.groupby("course")["exam_score"].mean()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

# Left: matplotlib defaults. Box on all four sides, both gridlines, a
# legend the bar colours already make redundant since each bar has its
# own label directly underneath it.
axes[0].bar(course_means.index, course_means.values, color=["#4477AA", "#EE6677", "#228833"])
axes[0].set_title("Before: matplotlib defaults")
axes[0].grid(True)

# Right: same data, decluttered. No top/right spines, no gridlines, the
# value printed directly on each bar instead of relying on the y-axis.
axes[1].bar(course_means.index, course_means.values, color="#4477AA")
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)
axes[1].set_yticks([])
for i, value in enumerate(course_means.values):
    axes[1].text(i, value + 1, f"{value:.0f}", ha="center")
axes[1].set_title("After: higher data-ink ratio")

fig.tight_layout()

<div style='background:#EAF3FA;border-left:5px solid #0369A1;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#0369A1;font-weight:bold'><i class="bi bi-info-circle-fill"></i> Key Concept: Data-ink ratio</span><br><br>
Every gridline, border, and redundant legend entry competes with your data for the reader's attention. The right-hand chart above removes the y-axis entirely (the labelled bars make it redundant) and the top/right spines, and prints each value once, directly on its bar, instead of forcing the reader to trace a line back to an axis. Decluttering is not about making a chart sparse for its own sake, it is about removing anything that does not carry information.
</div>

**One message per chart** is the last principle worth internalising before the chart-lie gallery in Section 2: a chart that tries to answer three questions at once usually answers none of them clearly. If you find yourself adding a third colour dimension, a secondary y-axis, and a trend line to the same chart, that is a sign you need two charts, not one crowded one.

## 2. Common Chart Lies

None of the charts below contain incorrect numbers. Each one is drawn in a way that makes the reader's brain compute the wrong conclusion from correct data. Recognising these on sight, in your own draft charts, is the actual skill.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))

axes[0].bar(course_means.index, course_means.values, color="#4477AA")
axes[0].set_ylim(0, 100)
axes[0].set_title("Honest: y-axis starts at 0")

axes[1].bar(course_means.index, course_means.values, color="#EE6677")
axes[1].set_ylim(65, 78)
axes[1].set_title("Misleading: y-axis starts at 65")

fig.tight_layout()

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-bug-fill"></i> Common Mistake: Truncating the y-axis</span><br><br>
The two charts above show the exact same three numbers. The right-hand one starts its y-axis at 65 instead of 0, which makes a 6-point difference between courses look like one course scores three times higher than another. Bar charts in particular must start at zero, because the reader judges bar height (and therefore a ratio) automatically. Line charts can sometimes justify a non-zero baseline if it is labelled clearly, bars almost never can.
</div>

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))

cohort_size = np.array([320, 340, 410, 580])
avg_score = np.array([71, 72, 70, 74])
years = ["2022", "2023", "2024", "2025"]

ax.plot(years, cohort_size, color="#4477AA", marker="o", label="Cohort size")
ax.set_ylabel("Cohort size", color="#4477AA")

ax2 = ax.twinx()
ax2.plot(years, avg_score, color="#EE6677", marker="o", label="Average score")
ax2.set_ylabel("Average score", color="#EE6677")

ax.set_title("Two independently scaled axes invite a false correlation");

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-bug-fill"></i> Common Mistake: Dual axes with independent scales</span><br><br>
Cohort size and average score both happen to rise together in this chart, but each axis was scaled independently to make the two lines fit nicely, which is exactly what makes dual axes dangerous: you can almost always pick scales that make two unrelated series appear to move together. If two series genuinely need comparing, plot their <b>percent change from a common baseline</b> on one shared axis instead, or use two separate charts stacked vertically with the same x-axis.
</div>

In [ ]:
enrollment = pd.Series(
    {"ML": 145, "Data Structures": 132, "Statistics": 98, "Databases": 41, "Networks": 28, "Ethics": 15}
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].pie(enrollment.values, labels=enrollment.index, autopct="%1.0f%%")
axes[0].set_title("Pie: hard to rank six similar slices")

axes[1].barh(enrollment.index[::-1], enrollment.values[::-1], color="#4477AA")
axes[1].set_title("Bar: ranking is immediate")

fig.tight_layout()

<div style='background:#FEF2F2;border-left:5px solid #DC2626;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#991B1B;font-weight:bold'><i class="bi bi-bug-fill"></i> Common Mistake: Too many pie slices</span><br><br>
Six categories is already past where a pie chart works: Ethics and Networks are visibly different bars, but as pie slices they are nearly impossible to rank by eye, exactly the angle-judgment weakness Cleveland and McGill measured. A horizontal bar chart, sorted by value, answers "which course has the most enrollment" instantly. Reserve pie charts for two or three categories where one slice is clearly dominant, and prefer a bar chart almost everywhere else.
</div>

## 3. When a Table Beats a Chart

A chart is for spotting a pattern at a glance. A table is for looking up an exact number, or comparing many numbers a reader needs to cite precisely, such as a results appendix someone will quote in a report. This project's `ark.plot.gt_style` module wraps `great_tables` with the same brand colours used everywhere else, so a results table looks like it belongs in the same document as the chart next to it:

In [ ]:
from great_tables import GT, md as gt_md

from ark.plot.gt_style import themed_gt

summary = results.groupby("course")["exam_score"].agg(mean="mean", std="std", n="count").round(1).reset_index()

table = themed_gt(
    GT(summary)
    .tab_header(title=gt_md("**Exam Score Summary**"), subtitle="Mean, spread, and sample size per course")
    .cols_label(course="Course", mean="Mean", std="Std. Dev.", n="N"),
    n_rows=len(summary),
)
table

<div style='background:#F4F5F6;border-left:5px solid #6B7280;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#6B7280;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: Pair a chart with a table, do not choose between them</span><br><br>
A results section in a report often wants both: the histogram from Part 5 to show the <i>shape</i> of the distribution, and a table like the one above to give the exact numbers someone will want to quote. Neither replaces the other.
</div>

## 4. From Default to Branded

Parts 5 and 6 used each library's own defaults on purpose, so the charts in those notebooks teach the mechanics without a styling layer in the way. This project's `ark.plot` module exists precisely so you do not have to restate the same colours, fonts, and spacing on every single chart by hand. Compare a lets-plot chart with and without it:

In [ ]:
from lets_plot import LetsPlot, aes, geom_histogram, gggrid, ggplot, ggsize

LetsPlot.setup_html()

default_chart = ggplot(results, aes(x="exam_score", fill="course")) + geom_histogram(bins=20, alpha=0.6)
default_chart + ggsize(450, 300)

In [ ]:
from lets_plot import scale_fill_manual

from ark.plot.theme import modern_theme, pro_colors

branded_chart = (
    ggplot(results, aes(x="exam_score", fill="course"))
    + geom_histogram(bins=20, alpha=0.85)
    + scale_fill_manual(values=pro_colors)
    + modern_theme(grid=True)
)
branded_chart + ggsize(450, 300)

`modern_theme()` is just one more layer added with `+`, exactly like every `geom_*()` in Part 6. `gggrid()` puts plots side by side in one output, which is the cleanest way to show a before/after comparison like this in a single cell:

In [ ]:
gggrid([default_chart + ggsize(380, 280), branded_chart + ggsize(380, 280)], ncol=2)

The matplotlib equivalent, `ark.plot.matplot_theme.configure_matplotlib_style()`, works differently: it updates matplotlib's global `rcParams`, so it applies to every chart drawn after you call it, not just one. That is the right trade-off for a notebook or script that draws many charts you want to look consistent, at the cost of not being able to mix styled and unstyled charts in the same figure:

In [ ]:
from ark.plot.matplot_theme import configure_matplotlib_style

configure_matplotlib_style(font_size=10, fig_width=5)

fig, ax = plt.subplots()
ax.hist(results["exam_score"], bins=20)
ax.set_xlabel("Exam score")
ax.set_ylabel("Number of students")
ax.set_title("Same chart as Part 5, after configure_matplotlib_style()");

<div style='background:#FFF1E6;border-left:5px solid #EA580C;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#9A3412;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Activity 1 - Brand a Boxplot</span><br><br>

<b>Goal:</b> Rebuild Part 5's seaborn boxplot of <code>exam_score</code> by <code>course</code>, but after calling <code>configure_matplotlib_style()</code> from this section. No other code changes: the same <code>sns.boxplot()</code> call should now pick up the brand colour cycle automatically.
<pre style='background:#FFF8E1;padding:10px;border-radius:4px;font-size:0.9em'>import seaborn as sns

fig, ax = plt.subplots()
sns.boxplot(data=results, x="course", y="exam_score", ax=ax)
ax.set_title(...)</pre>
</div>

In [ ]:
import seaborn as sns

# TODO: boxplot of exam_score by course, styled by the already-active house theme
...

<div style='background:#F4F5F6;border-left:5px solid #6B7280;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#6B7280;font-weight:bold'><i class="bi bi-lightbulb-fill"></i> Pro Tip: A house theme is a contract, not a one-off setting</span><br><br>
The value of <code>ark.plot</code> is not any single colour choice, it is that every chart in this project, in any notebook, in any report, reaches for the same module instead of redefining its own palette. When the brand colours change, they change in one file.
</div>

## Capstone: Rescue a Misleading Chart

Below is a chart with three problems from this notebook stacked together: a truncated y-axis, no axis labels, and matplotlib's unbranded defaults. Fix all three, then add one caption sentence stating the chart's single message.

In [ ]:
# The chart to rescue. Run this first to see what is wrong with it.
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(course_means.index, course_means.values, color=["#4477AA", "#EE6677", "#228833"])
ax.set_ylim(65, 78)
fig.tight_layout()

<div style='background:#FFF1E6;border-left:5px solid #EA580C;padding:14px 18px;border-radius:6px;margin:16px 0'>
<span style='color:#9A3412;font-weight:bold'><i class="bi bi-puzzle-fill"></i> Capstone Exercise - Fix the Chart</span><br><br>

<b>Goal:</b>
<ol>
<li>Fix the y-axis to start at 0</li>
<li>Add an x and y axis label</li>
<li>Apply the house style with <code>configure_matplotlib_style()</code></li>
<li>Add one print statement with a one-sentence caption stating the chart's single message</li>
</ol>
<b>Hint:</b> <code>configure_matplotlib_style()</code> only affects charts drawn after it runs, so call it before <code>plt.subplots()</code>, not after.
</div>

In [ ]:
# TODO: rescue the chart
...

print("Caption: ...")

## Summary

| Concept | Key rule |
|---|---|
| Chart selection | Let the question pick the chart; position beats angle beats area for accurate reading |
| Data-ink ratio | Every gridline, border, and redundant legend competes with your data |
| One message per chart | A third colour dimension or a second y-axis is a sign you need two charts |
| Truncated axes | Bar charts must start at 0; a non-zero baseline exaggerates differences |
| Dual axes | Independent scales can make any two series look correlated |
| Pie charts | Fine for 2-3 dominant slices, a sorted bar chart almost everywhere else |
| Tables vs. charts | Charts for patterns, tables for numbers a reader will quote exactly |
| `ark.plot` house style | One module, not one styling decision repeated on every chart |

### Further reading

- Edward Tufte, *The Visual Display of Quantitative Information* - the original case for data-ink ratio
- William Cleveland and Robert McGill, "Graphical Perception: Theory, Experimentation, and Application to the Development of Graphical Methods" (1984) - the perceptual ranking behind "position beats angle"
- Stephen Few, *Show Me the Numbers* - practical chart and dashboard design
- Alberto Cairo, *The Truthful Art* - visualisation ethics and how charts deceive
- Cole Nussbaumer Knaflic, *Storytelling with Data* - the most widely read practitioner book on this topic
- Claus Wilke, *Fundamentals of Data Visualization* (free online) - a modern, ggplot-oriented treatment, directly relevant given lets-plot's ggplot heritage
- [Financial Times Visual Vocabulary](https://ft-interactive.github.io/visual-vocabulary/) - a practical chart-type-selection reference

**This completes the data visualisation series within Part 1: Python Basics.** The next tutorials folder, `02-dev-tools`, covers the engineering practices used from here on, once it gets the same rewrite treatment as Parts 1-7.